In [4]:
! pip install torchinfo

In [15]:
%cd MielPops/

/content/MielPops


In [14]:
!pwd

/content


In [16]:
import torch
import torch.nn as nn
import torchinfo
from model.mini_cnn import MiniCNN
from result.repondeur import prediction_to_csv
import torchvision
import numpy as np
import csv
from tqdm import tqdm
from model.loader import load_dataset, load_test_loader

In [17]:
# from google.colab import drive
# drive.mount('/content/drive')

In [18]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU - training will be slow!")

✓ Using GPU: Tesla T4


In [19]:
def train_crossEntropyLoss(train_set, model, epochs):
    """
    Entraine un model à partir d'un train_set donné. Retourne les metrics de l'entrainement
    """

    # Parameters
    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.03,
        momentum=0.9,
        weight_decay=5e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=200
    )


    accuracies = np.zeros(epochs)
    losses = np.zeros(epochs)

    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(
            train_set,
            desc=f"Model training | Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            #prediction
            logits = model(images)
            loss = criterion(logits, labels)

            #propagation du gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Calcul des metriques
            total_loss += loss.item()

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "accuracy": f"{correct/total*100:.2f}%"
            })
        accuracies[epoch] = correct/total
        losses[epoch] = total_loss

        scheduler.step()
    return accuracies, losses

In [20]:
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from utils.CustomDataset import CustomDataset
from utils.CustomSkibidiDataset import CustomSkibidiDataset

from torchvision.transforms import v2

In [21]:
# 1. Define Image Transforms
# HUUUUGOOOOOOOO
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     torchvision.transforms.functional.to_dtype(float16, scale=True),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = v2.Compose([
    #v2.ToImage(),                  # Convert to Tensor (if it's PIL)
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True), # Scales to float16 [0, 1] # TODO
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# 2. Instantiate your Dataset
"""train_dataset = CustomDataset(
    img_dir='data/',
    annotations_file='data/train.csv',
    transform=transform,
    target_transform=None # ???
)"""

skibidi_dataset = CustomSkibidiDataset(
    img_dir='data/treated_train/',
    csv_mapper='data/treated_train.csv',
    transform=transform,
    target_transform=None # ???
)

train_size = int(0.8 * len(skibidi_dataset))
test_size = len(skibidi_dataset) - train_size

train_dataset, validation_dataset = random_split(skibidi_dataset, [train_size, test_size])

Found folders ['Andrena ventralis', 'Andrena plana', 'Andrena vicinoides', 'Andrena mediovittata', 'Andrena wilkella', 'Andrena cineraria', 'Andrena orbitalis', 'Andrena vaga', 'Andrena rudbeckiae', 'Andrena carantonica', 'Andrena chengtehensis', 'Andrena melanochroa', 'Andrena hattorfiana', 'Andrena banksi', 'Andrena hesperia', 'Andrena barbilabris', 'Andrena aliciae', 'Andrena fulvata', 'Andrena haemorrhoa', 'Andrena rufula', 'Andrena crawfordi', 'Andrena lineolata', 'Andrena irana', 'Andrena fulva', 'Andrena angustitarsata', 'Andrena dilleri', 'Andrena aerinifrons', 'Andrena discors', 'Andrena flavipes', 'Andrena villipes', 'Andrena denticulata', 'Andrena afrensis', 'Andrena fortipunctata', 'Andrena limbata', 'Andrena pinguis', 'Andrena clarkella', 'Andrena costillensis', 'Andrena convallaria', 'Andrena dorsata', 'Andrena leucophaea', 'Andrena barbareae', 'Andrena florivaga', 'Andrena nitida', 'Andrena bicolor', 'Andrena mendica', 'Andrena vulpecula', 'Andrena perimelas', 'Andrena n

In [42]:
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
import timm

In [73]:
#set up training
epochs = 5

# convnext tiny
"""
# 1. Use Weights instead of a boolean (modern torchvision style)
# weights = ConvNeXt_Tiny_Weights.DEFAULT
# model = convnext_tiny(weights=weights)

# 2. Freeze parameters ONLY if using pretrained weights
# for param in model.parameters():
#     param.requires_grad = False
# Unfreeze the last stage (Stage 3) for better adaptation
# for param in model.features[7].parameters():
#     param.requires_grad = True

# 3. Correctly access the classifier head
# ConvNeXt's head is a Sequential: (0): LayerNorm, (1): Flatten, (2): Linear
in_features = model.classifier[2].in_features
num_classes = 50
model.classifier[2] = torch.nn.Linear(in_features, num_classes)

"""

# convnext atto

model = timm.create_model('convnext_nano.in12k', pretrained=True, num_classes=50)
# 2. Freezing the backbone
# In timm, the classifier is usually stored in 'model.head'
for name, param in model.named_parameters():
    if "head" not in name :#and "stages.3" not in name :
        param.requires_grad = False
        #print("u")
    else :
        print(name)




# 4. Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_set = DataLoader(
    dataset=train_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

validation_test = DataLoader(
    dataset=validation_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

head.norm.weight
head.norm.bias
head.fc.weight
head.fc.bias


In [64]:
# for name, param in model.named_parameters():
#   print(name)
#   if "head" not in name :
#     pass
#   else :
#     print("GAUGH")

stem.0.weight
stem.0.bias
stem.1.weight
stem.1.bias
stages.0.blocks.0.gamma
stages.0.blocks.0.conv_dw.weight
stages.0.blocks.0.conv_dw.bias
stages.0.blocks.0.norm.weight
stages.0.blocks.0.norm.bias
stages.0.blocks.0.mlp.fc1.weight
stages.0.blocks.0.mlp.fc1.bias
stages.0.blocks.0.mlp.fc2.weight
stages.0.blocks.0.mlp.fc2.bias
stages.0.blocks.1.gamma
stages.0.blocks.1.conv_dw.weight
stages.0.blocks.1.conv_dw.bias
stages.0.blocks.1.norm.weight
stages.0.blocks.1.norm.bias
stages.0.blocks.1.mlp.fc1.weight
stages.0.blocks.1.mlp.fc1.bias
stages.0.blocks.1.mlp.fc2.weight
stages.0.blocks.1.mlp.fc2.bias
stages.1.downsample.0.weight
stages.1.downsample.0.bias
stages.1.downsample.1.weight
stages.1.downsample.1.bias
stages.1.blocks.0.gamma
stages.1.blocks.0.conv_dw.weight
stages.1.blocks.0.conv_dw.bias
stages.1.blocks.0.norm.weight
stages.1.blocks.0.norm.bias
stages.1.blocks.0.mlp.fc1.weight
stages.1.blocks.0.mlp.fc1.bias
stages.1.blocks.0.mlp.fc2.weight
stages.1.blocks.0.mlp.fc2.bias
stages.1.block

In [74]:
#training
training_curves, training_curves = train_crossEntropyLoss(train_set,model, epochs)

Model training | Epoch 1/5: 100%|██████████| 656/656 [01:16<00:00,  8.56it/s, batch_loss=0.4720, accuracy=76.56%]
Model training | Epoch 2/5: 100%|██████████| 656/656 [01:14<00:00,  8.75it/s, batch_loss=0.2333, accuracy=89.81%]
Model training | Epoch 3/5: 100%|██████████| 656/656 [01:15<00:00,  8.73it/s, batch_loss=0.2612, accuracy=91.52%]
Model training | Epoch 4/5: 100%|██████████| 656/656 [01:14<00:00,  8.75it/s, batch_loss=0.3215, accuracy=91.70%]
Model training | Epoch 5/5: 100%|██████████| 656/656 [01:15<00:00,  8.71it/s, batch_loss=0.8053, accuracy=91.69%]


In [75]:
torch.save(model.state_dict(), f"model/Convnext_Nano_{epochs}_epochs.pth")

In [47]:
import csv

In [52]:
def prediction_to_csv(validation_dataset,model) :
    """
    Charge un dataset et retourne le csv des predictions associé.
    """

    model = model.to(device)
    model.eval()

    id = 0

    # enregistre les predictions
    predictions_array = []

    with torch.no_grad() :

        pbar = tqdm(
            validation_dataset,
            desc="Inference"
        )

        for images, labels in pbar :

            images = images.to(device) # a modifier en fonction du test_set
            img_logits = model(images)


            predictions = img_logits.argmax(dim=1)
            predictions = predictions.cpu().numpy()

            for pred, label in zip(predictions, labels):
                id += 1
                predictions_array.append({
                    "id": id,
                    "pred": int(pred),
                    "true": int(label)
                })

        output_path = f"prediction.csv"

        # Sauvegarde CSV
        with open(output_path, "w", newline='', encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames = ["id", "pred", "true"])
            writer.writeheader()
            writer.writerows(predictions_array)
        print(f"Les predictions ont été sauvegardees dans {output_path}.")

In [53]:
prediction_to_csv(validation_test, model)

Inference:   0%|          | 0/164 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Inference: 100%|██████████| 164/164 [00:12<00:00, 13.21it/s]

Les predictions ont été sauvegardees dans prediction.csv.


In [50]:
import pandas as pd
from eval.Analyser import Analyser

Classes: ['Andrena aerinifrons', 'Andrena afrensis', 'Andrena aliciae', 'Andrena angustitarsata', 'Andrena banksi', 'Andrena barbareae', 'Andrena barbilabris', 'Andrena bicolor', 'Andrena carantonica', 'Andrena chengtehensis', 'Andrena cineraria', 'Andrena clarkella', 'Andrena convallaria', 'Andrena costillensis', 'Andrena crawfordi', 'Andrena denticulata', 'Andrena dilleri', 'Andrena discors', 'Andrena dorsata', 'Andrena flavipes', 'Andrena florivaga', 'Andrena fortipunctata', 'Andrena fulva', 'Andrena fulvata', 'Andrena haemorrhoa', 'Andrena hattorfiana', 'Andrena hesperia', 'Andrena irana', 'Andrena leucophaea', 'Andrena limbata', 'Andrena lineolata', 'Andrena mediovittata', 'Andrena melanochroa', 'Andrena mendica', 'Andrena nigroaenea', 'Andrena nitida', 'Andrena orbitalis', 'Andrena perimelas', 'Andrena perplexa', 'Andrena pinguis', 'Andrena plana', 'Andrena prodigiosa', 'Andrena rudbeckiae', 'Andrena rufula', 'Andrena vaga', 'Andrena ventralis', 'Andrena vicinoides', 'Andrena vil

In [51]:
data = pd.read_csv("prediction.csv")

analyser = Analyser(data)
report = analyser.generate_report()
print(report)

{'f1_score_avg': 0.7211398335706687, 'f1_score_per_class': array([0.98850575, 0.1884058 , 0.99300699, 0.86549708, 0.44      ,
       0.85714286, 0.90697674, 0.95283019, 0.69767442, 0.96      ,
       0.8496732 , 0.31351351, 0.9047619 , 0.77637131, 0.83817427,
       0.83333333, 0.52631579, 0.39662447, 0.78929766, 0.96969697,
       0.7972028 , 0.65402844, 0.84429066, 0.37344398, 0.62978723,
       0.9137931 , 0.35714286, 0.91707317, 0.97619048, 0.97991968,
       0.27586207, 0.91472868, 0.98850575, 0.94936709, 0.36158192,
       0.90163934, 0.82485876, 0.85211268, 0.65454545, 0.44239631,
       0.79545455, 0.3452381 , 0.78195489, 0.7518797 , 0.90361446,
       0.7751938 , 0.57738095]), 'best_f1': np.int64(2)}
